In [ ]:
# Preprocesamiento de Datos con Machine Learning
## Dataset: Breast Cancer Wisconsin

"""
OBJETIVO DEL NOTEBOOK:
Este notebook es una guía instructiva sobre el preprocesamiento de datos,
una etapa crucial en cualquier proyecto de Machine Learning.

Utilizaremos el dataset Breast Cancer Wisconsin para demostrar las mejores
prácticas en limpieza, exploración y preparación de datos.
"""

# ============================================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ============================================================================
print("=" * 80)
print("FASE 1: IMPORTACIÓN DE LIBRERÍAS")
print("=" * 80)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)

print("✓ Librerías importadas correctamente\n")

# ============================================================================
# 2. CARGA Y EXPLORACIÓN INICIAL DEL DATASET
# ============================================================================
print("=" * 80)
print("FASE 2: CARGA Y EXPLORACIÓN INICIAL")
print("=" * 80)

# Cargar el dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print("📊 INFORMACIÓN DEL DATASET:")
print(f"- Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"- Características (features): {df.shape[1] - 1}")
print(f"- Variable objetivo: {data.target_names}")
print(f"\n📝 Descripción del dataset:")
print("El dataset contiene medidas de células de tumores de mama.")
print("Target: 0 = Maligno, 1 = Benigno\n")

# Primeras filas
print("Primeras 5 filas del dataset:")
print(df.head())

# ============================================================================
# 3. ANÁLISIS EXPLORATORIO DE DATOS (EDA)
# ============================================================================
print("\n" + "=" * 80)
print("FASE 3: ANÁLISIS EXPLORATORIO DE DATOS (EDA)")
print("=" * 80)

# 3.1 Información general
print("\n📋 INFORMACIÓN GENERAL:")
print(df.info())

# 3.2 Estadísticas descriptivas
print("\n📈 ESTADÍSTICAS DESCRIPTIVAS:")
print(df.describe())

# 3.3 Verificar valores faltantes
print("\n🔍 VALORES FALTANTES:")
missing = df.isnull().sum()
if missing.sum() == 0:
    print("✓ No hay valores faltantes en el dataset")
else:
    print(missing[missing > 0])

# 3.4 Verificar duplicados
duplicados = df.duplicated().sum()
print(f"\n🔄 REGISTROS DUPLICADOS: {duplicados}")
if duplicados == 0:
    print("✓ No hay registros duplicados")

# 3.5 Distribución de la variable objetivo
print("\n🎯 DISTRIBUCIÓN DE LA VARIABLE OBJETIVO:")
target_counts = df['target'].value_counts()
print(target_counts)
print(f"\nPorcentaje:")
print(f"- Benignos (1): {target_counts[1]/len(df)*100:.2f}%")
print(f"- Malignos (0): {target_counts[0]/len(df)*100:.2f}%")

# Visualización de la distribución
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
axes[0].bar(['Maligno', 'Benigno'], target_counts.values, color=['#ff6b6b', '#4ecdc4'])
axes[0].set_title('Distribución de Diagnósticos', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Frecuencia')
axes[0].grid(axis='y', alpha=0.3)

# Gráfico de pastel
axes[1].pie(target_counts.values, labels=['Maligno', 'Benigno'], 
            autopct='%1.1f%%', startangle=90, colors=['#ff6b6b', '#4ecdc4'])
axes[1].set_title('Proporción de Diagnósticos', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# ============================================================================
# 4. ANÁLISIS DE DISTRIBUCIONES Y OUTLIERS
# ============================================================================
print("\n" + "=" * 80)
print("FASE 4: ANÁLISIS DE DISTRIBUCIONES Y OUTLIERS")
print("=" * 80)

# Seleccionar algunas características importantes para visualizar
features_importantes = ['mean radius', 'mean texture', 'mean perimeter', 
                        'mean area', 'mean smoothness', 'mean compactness']

# 4.1 Visualizar distribuciones
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, feature in enumerate(features_importantes):
    axes[idx].hist(df[feature], bins=30, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribución: {feature}', fontweight='bold')
    axes[idx].set_xlabel('Valor')
    axes[idx].set_ylabel('Frecuencia')
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 4.2 Detección de outliers con boxplots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, feature in enumerate(features_importantes):
    bp = axes[idx].boxplot(df[feature], patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('#3498db')
    axes[idx].set_title(f'Boxplot: {feature}', fontweight='bold')
    axes[idx].set_ylabel('Valor')
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 4.3 Identificar outliers usando IQR
print("\n📊 ANÁLISIS DE OUTLIERS (Método IQR):")
for feature in features_importantes:
    Q1 = df[feature].quantile(0.25)
    Q3 = df[feature].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[feature] < Q1 - 1.5 * IQR) | (df[feature] > Q3 + 1.5 * IQR)]
    print(f"{feature}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.2f}%)")

# ============================================================================
# 5. ANÁLISIS DE CORRELACIÓN
# ============================================================================
print("\n" + "=" * 80)
print("FASE 5: ANÁLISIS DE CORRELACIÓN")
print("=" * 80)

# Matriz de correlación
correlation_matrix = df.corr()

# Visualizar matriz de correlación (solo primeras 10 features para claridad)
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix.iloc[:10, :10], annot=True, fmt='.2f', 
            cmap='coolwarm', center=0, square=True, linewidths=1)
plt.title('Matriz de Correlación (10 primeras características)', 
          fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Identificar características altamente correlacionadas
print("\n🔗 CARACTERÍSTICAS ALTAMENTE CORRELACIONADAS (|r| > 0.9):")
high_corr = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.9:
            high_corr.append({
                'Feature 1': correlation_matrix.columns[i],
                'Feature 2': correlation_matrix.columns[j],
                'Correlación': correlation_matrix.iloc[i, j]
            })

if high_corr:
    corr_df = pd.DataFrame(high_corr)
    print(corr_df.to_string(index=False))
    print("\n⚠️ Considerar eliminar características redundantes")
else:
    print("No hay características con correlación > 0.9")

# ============================================================================
# 6. PREPROCESAMIENTO: SEPARACIÓN DE DATOS
# ============================================================================
print("\n" + "=" * 80)
print("FASE 6: SEPARACIÓN DE DATOS (TRAIN/TEST)")
print("=" * 80)

# Separar características y variable objetivo
X = df.drop('target', axis=1)
y = df['target']

# División train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✓ Datos divididos:")
print(f"  - Entrenamiento: {X_train.shape[0]} muestras ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"  - Prueba: {X_test.shape[0]} muestras ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\n✓ Distribución estratificada mantenida:")
print(f"  - Train: Benignos={sum(y_train)/len(y_train)*100:.1f}%")
print(f"  - Test: Benignos={sum(y_test)/len(y_test)*100:.1f}%")

# ============================================================================
# 7. PREPROCESAMIENTO: ESCALADO DE CARACTERÍSTICAS
# ============================================================================
print("\n" + "=" * 80)
print("FASE 7: ESCALADO DE CARACTERÍSTICAS")
print("=" * 80)

print("\n📚 TIPOS DE ESCALADORES:")
print("""
1. StandardScaler: Estandarización (media=0, std=1)
   - Útil cuando los datos siguen distribución normal
   - Sensible a outliers
   
2. MinMaxScaler: Normalización (rango 0-1)
   - Útil cuando necesitas valores en rango específico
   - Sensible a outliers
   
3. RobustScaler: Usa mediana e IQR
   - Robusto ante outliers
   - Recomendado cuando hay muchos outliers
""")

# Aplicar StandardScaler (el más común)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convertir de nuevo a DataFrame para análisis
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)

print("\n✓ Escalado completado con StandardScaler")
print("\n📊 COMPARACIÓN ANTES/DESPUÉS DEL ESCALADO:")
print("\nANTES (primeras 3 características):")
print(X_train.iloc[:3, :3])
print("\nDESPUÉS (primeras 3 características):")
print(X_train_scaled_df.iloc[:3, :3])

# Visualizar el efecto del escalado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Antes del escalado
axes[0].boxplot([X_train.iloc[:, i] for i in range(5)], 
                labels=X.columns[:5])
axes[0].set_title('ANTES del Escalado', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Valor')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(alpha=0.3)

# Después del escalado
axes[1].boxplot([X_train_scaled_df.iloc[:, i] for i in range(5)], 
                labels=X.columns[:5])
axes[1].set_title('DESPUÉS del Escalado', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Valor estandarizado')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================================
# 8. REDUCCIÓN DE DIMENSIONALIDAD CON PCA
# ============================================================================
print("\n" + "=" * 80)
print("FASE 8: REDUCCIÓN DE DIMENSIONALIDAD (PCA)")
print("=" * 80)

print("\n📚 PCA (Principal Component Analysis):")
print("""
- Reduce el número de características manteniendo la mayor varianza
- Útil para visualización y reducir sobreajuste
- Crea características no correlacionadas
""")

# Aplicar PCA
pca = PCA()
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Varianza explicada
varianza_explicada = pca.explained_variance_ratio_
varianza_acumulada = np.cumsum(varianza_explicada)

print(f"\n✓ Varianza explicada por componente:")
for i in range(min(10, len(varianza_explicada))):
    print(f"  PC{i+1}: {varianza_explicada[i]*100:.2f}% "
          f"(Acumulada: {varianza_acumulada[i]*100:.2f}%)")

# Visualizar varianza explicada
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
axes[0].bar(range(1, min(16, len(varianza_explicada)+1)), 
            varianza_explicada[:15], alpha=0.7, color='#3498db')
axes[0].set_xlabel('Componente Principal')
axes[0].set_ylabel('Varianza Explicada')
axes[0].set_title('Varianza Explicada por Componente', fontweight='bold')
axes[0].grid(alpha=0.3)

# Gráfico acumulado
axes[1].plot(range(1, len(varianza_acumulada)+1), 
             varianza_acumulada, marker='o', linestyle='-', linewidth=2)
axes[1].axhline(y=0.95, color='r', linestyle='--', label='95% varianza')
axes[1].axhline(y=0.99, color='g', linestyle='--', label='99% varianza')
axes[1].set_xlabel('Número de Componentes')
axes[1].set_ylabel('Varianza Acumulada')
axes[1].set_title('Varianza Explicada Acumulada', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Número óptimo de componentes
n_componentes_95 = np.argmax(varianza_acumulada >= 0.95) + 1
n_componentes_99 = np.argmax(varianza_acumulada >= 0.99) + 1

print(f"\n💡 RECOMENDACIÓN:")
print(f"  - Para mantener 95% de varianza: {n_componentes_95} componentes")
print(f"  - Para mantener 99% de varianza: {n_componentes_99} componentes")
print(f"  - Reducción: de {X_train.shape[1]} a {n_componentes_95} características")

# Visualización en 2D con PCA
pca_2d = PCA(n_components=2)
X_train_2d = pca_2d.fit_transform(X_train_scaled)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_train_2d[:, 0], X_train_2d[:, 1], 
                     c=y_train, cmap='coolwarm', alpha=0.6, s=50)
plt.xlabel('Primera Componente Principal', fontsize=12)
plt.ylabel('Segunda Componente Principal', fontsize=12)
plt.title('Visualización del Dataset en 2D con PCA', 
          fontsize=14, fontweight='bold', pad=20)
plt.colorbar(scatter, label='Diagnóstico (0=Maligno, 1=Benigno)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================================
# 9. RESUMEN DEL PREPROCESAMIENTO
# ============================================================================
print("\n" + "=" * 80)
print("FASE 9: RESUMEN DEL PREPROCESAMIENTO")
print("=" * 80)

print("""
✅ PASOS COMPLETADOS:

1. ✓ Carga y exploración del dataset
   - 569 muestras, 30 características
   - Sin valores faltantes ni duplicados
   
2. ✓ Análisis exploratorio (EDA)
   - Distribución de clases balanceada (63% benigno)
   - Identificación de outliers
   - Análisis de correlaciones
   
3. ✓ Separación de datos
   - Train: 80% | Test: 20%
   - Estratificación mantenida
   
4. ✓ Escalado de características
   - StandardScaler aplicado
   - Media=0, Desviación=1
   
5. ✓ Reducción de dimensionalidad
   - PCA aplicado
   - 95% varianza con menos componentes

📊 DATOS LISTOS PARA MODELADO:
""")

print(f"\nConjunto de Entrenamiento:")
print(f"  - Shape original: {X_train.shape}")
print(f"  - Shape escalado: {X_train_scaled.shape}")
print(f"  - Shape PCA (2D): {X_train_2d.shape}")

print(f"\nConjunto de Prueba:")
print(f"  - Shape original: {X_test.shape}")
print(f"  - Shape escalado: {X_test_scaled.shape}")

print("""
🎯 PRÓXIMOS PASOS:
   1. Entrenar modelos de clasificación
   2. Evaluar rendimiento con métricas
   3. Optimizar hiperparámetros
   4. Validación cruzada
""")

# ============================================================================
# 10. EXPORTAR DATOS PREPROCESADOS (OPCIONAL)
# ============================================================================
print("\n" + "=" * 80)
print("DATOS PREPROCESADOS DISPONIBLES EN MEMORIA")
print("=" * 80)
print("""
Variables disponibles para modelado:
- X_train, X_test: Datos originales
- X_train_scaled, X_test_scaled: Datos escalados (recomendado)
- X_train_pca, X_test_pca: Datos con PCA
- y_train, y_test: Variables objetivo
- scaler: Objeto StandardScaler ajustado
- pca: Objeto PCA ajustado
""")